In [6]:
import torch 


### Computing gradients via autograde

In [7]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b 
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a,y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True) 
"""
By default, PyTorch destroys 
the computation graph after 
calculating the gradients to 
free memory. However, since 
we will reuse this 
computation graph shortly, 
we set retain_graph=True 
so that it stays in memory

"""



print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [8]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


### Implementing multilayer neural networks

In [10]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            
            #1st hidden layer 
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            #2nd hidden layer 
            torch.nn.Linear(30,20),
            torch.nn.ReLU(),

            #output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


model = NeuralNetwork(50,3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [21]:
lr_1 = (50*30+30)  # 1st layer paramerts
lr_2 = (30*20+20) #2nd layer paramerts 
lr_3 =  (20*3+3) # output layer paramerts 

total_number_parameters = lr_1 + lr_2 + lr_3

print(f"layer 1 paramerts: {lr_1}")
print(f"layer 2 paramerts: {lr_2}")
print(f"layer 3 paramerts: {lr_3}")
print(f"total_number_parameters: {total_number_parameters}")

layer 1 paramerts: 1530
layer 2 paramerts: 620
layer 3 paramerts: 63
total_number_parameters: 2213


In [12]:
# total number of parameters

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable parameters: {num_params}")

Total number of trainable parameters: 2213


In [19]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0141, -0.0508,  0.0994,  ..., -0.0313,  0.1402,  0.0358],
        [-0.0711, -0.0843, -0.0012,  ...,  0.0019,  0.0800,  0.0686],
        [ 0.0763, -0.1193, -0.1146,  ..., -0.1156, -0.0755, -0.0773],
        ...,
        [-0.0268,  0.0913, -0.0173,  ...,  0.1122,  0.0079,  0.0873],
        [ 0.0523,  0.0387,  0.0875,  ...,  0.0884, -0.0989, -0.0336],
        [-0.1406,  0.0634,  0.0843,  ...,  0.0045,  0.0190,  0.0675]],
       requires_grad=True)

In [22]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [23]:
model.layers[0].bias.shape

torch.Size([30])

In [26]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[ 0.0718,  0.2009, -0.0332]], grad_fn=<AddmmBackward0>)


```bash
The <AddmmBackward0> part of grad_fn= <AddmmBackward0> specifies the
operation performed. In this case, it is an Addmm operation. Addmm stands for matrix
multiplication (mm) followed by an addition (Add)

In [25]:
with torch.no_grad(): #  This tells PyTorch that it doesn’t need to keep track of the gradients,
#                    which can result in significant savings in memory and computation
    out = model(X)
print(out)

tensor([[ 0.0718,  0.2009, -0.0332]])


In [27]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3292, 0.3745, 0.2963]])


### Setting up efficient data loaders

-  The DataLoader handles how the data is shuffled and assembled into batches

####  Creating a small toy dataset

In [28]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

In [29]:
# Defining a custom Dataset class 

from torch.utils.data import Dataset 

class ToyDataset(Dataset):
    def __init__(self, X,y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]

        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train,y_train)
test_ds = ToyDataset(X_test,y_test)

In [32]:
len(test_ds)

2

In [34]:
# Instantiating data loaders

from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0
)

test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False,
    num_workers = 0
)

In [35]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [36]:
# A training loader that drops the last batch 
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0, #  for parallelizing data loading
    drop_last=True
)

for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


### training loop

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs = 2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3 
for epoch in range(num_epochs):

    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad() # To prevent undesired gradient accumulation
        loss.backward()
        optimizer.step()

        # LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

    model.eval()
    

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [42]:
(2*30+30) + (30*20+20) + (20*2+2)

752

In [41]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable parameters: {num_params}")

Total number of trainable parameters: 752


In [43]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [44]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [45]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [46]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [50]:
# A function to compute the prediction accuracy

def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0 


    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct+= torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [51]:
compute_accuracy(model, train_loader)

1.0

In [52]:
##  Saving and loading models

torch.save(model.state_dict(), "model.pth")

## Trainin on GPU

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [53]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs = 2, num_outputs=2).to(device)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3 
for epoch in range(num_epochs):

    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device) 
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad() # To prevent undesired gradient accumulation
        loss.backward()
        optimizer.step()

        # LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

    model.eval()
    

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [58]:
a = torch.rand((50,50))
b = torch.rand((50,50))

%timeit a @ b

12.5 μs ± 2 μs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [59]:
%timeit a.to(device) @ b.to(device)

236 μs ± 52.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
